# Load Libraries and packages

In [6]:
import pandas as pd
import csv
import os
import sys

In [ ]:
#which crop to run this notebook for
crop = "soy"  # "corn" or "soy" — must match whichever pair is uncommented below

#uncomment the pair that matches `crop` above, comment out the other
path_df_step1 = "../01b_yield_data_soy/created_dfs_step1/"       #"../01a_yield_data_corn/created_dfs_step1/"
filename = "df_soy_yield_2026.csv"                                  #"df_corn_yield_2026.csv"
save_file = "created_dfs_step2/df_yield_climdiv_soy_paper.csv"      #"created_dfs_step2/df_yield_climdiv_corn_paper.csv"

#safety check: make sure crop, path_df_step1, filename, and save_file all actually agree,
#so a half-finished comment-toggle (only some of the three lines swapped) gets caught immediately
#instead of silently producing e.g. a "corn" output built from soy data.
other_crop = "corn" if crop == "soy" else "soy"
for label, value in [("path_df_step1", path_df_step1), ("filename", filename), ("save_file", save_file)]:
    if other_crop in value.lower():
        raise ValueError(f"crop = '{crop}' but {label} = '{value}' contains '{other_crop}' — looks like only some of the corn/soy lines got toggled.")
    if crop not in value.lower():
        raise ValueError(f"crop = '{crop}' but {label} = '{value}' does not contain '{crop}' — looks like only some of the corn/soy lines got toggled.")

print(f"Crop selection OK: crop='{crop}'")
print(f"  path_df_step1 = {path_df_step1}")
print(f"  filename      = {filename}")
print(f"  save_file     = {save_file}")

#### Weather data

In [ ]:
#-----------------------------------------------------------------
#1) Read the county-to-climdivs mapping file into a lookup dict
#the file has 3 columns (POSTAL_FIPS_ID, NCDC_FIPS_ID, CLIMDIV_ID)
#we'll map NCDC_FIPS_ID -> (POSTAL_FIPS_ID, CLIMDIV_ID)
#-----------------------------------------------------------------

mapping = {}
with open(os.path.join("extracted_noaa_climdiv_data", "county-to-climdivs.txt"), "r") as f:
    next(f)  #skip header line if it exists
    for line in f:
        parts = line.strip().split()
        if len(parts) != 3:
            continue
        postal_fips, ncdc_fips, climdiv_id = parts
        mapping[ncdc_fips] = (postal_fips, climdiv_id)

#-----------------------------------------------------------------
#2) Define a helper function to parse each line in tmaxcy/pcpncy
#-----------------------------------------------------------------
def parse_clim_line(line):
    """
    Given a line (string) from tmaxcy or pcpncy,
    returns a dict with raw_code, state, county, division, year, and the 12 monthly values.
    If the line can't be mapped (NCDC FIPS not found), return None.
    """
    parts = line.strip().split()
    if len(parts) < 13:
        return None  #not enough data
    
    #the first item is the 11-digit code: e.g. "01001271895"
    code = parts[0]
    monthly_values = parts[1:]  #the next 12 numbers
    
    ncdc_fips = code[:5]       #first 5 digits
    data_type = code[5:7]      #next 2 digits (27 for tmax, 01 for pcpn)
    year = code[7:]            #last 4 digits
    
    if ncdc_fips not in mapping:
        return None
    
    postal_fips, climdiv_id = mapping[ncdc_fips]
    #postal_fips e.g. "04001" => correct_state="04", correct_county="001"
    correct_state = postal_fips[:2]
    correct_county = postal_fips[2:]
    #climdiv_id e.g. "0202" => last two digits "02" for division
    division = climdiv_id[-2:]
    
    #create a dict with the data, including the raw_code for clarity
    return {
        "raw_code": code,
        "state": correct_state,
        "county": correct_county,
        "division": division,
        "year": year,
        "Jan": monthly_values[0],
        "Feb": monthly_values[1],
        "Mar": monthly_values[2],
        "Apr": monthly_values[3],
        "May": monthly_values[4],
        "Jun": monthly_values[5],
        "Jul": monthly_values[6],
        "Aug": monthly_values[7],
        "Sep": monthly_values[8],
        "Oct": monthly_values[9],
        "Nov": monthly_values[10],
        "Dec": monthly_values[11]
    }

#-----------------------------------------------------------------
#3) Read & parse tmaxcy (temperature) lines
#-----------------------------------------------------------------
tmax_records = []
with open(os.path.join("extracted_noaa_climdiv_data", "climdiv-tmaxcy-v1.0.0-20260806.txt"), "r") as f:
    for line in f:
        parsed = parse_clim_line(line)
        if parsed:
            tmax_records.append(parsed)

df_tmax = pd.DataFrame(tmax_records)
print("TMAX DataFrame:\n", df_tmax.head())

#-----------------------------------------------------------------
#4) Lastly Read & parse pcpncy (precipitation) lines
#-----------------------------------------------------------------
pcpn_records = []
with open(os.path.join("extracted_noaa_climdiv_data", "climdiv-pcpncy-v1.0.0-20260806.txt"), "r") as f:
    for line in f:
        parsed = parse_clim_line(line)
        if parsed:
            pcpn_records.append(parsed)

df_pcpn = pd.DataFrame(pcpn_records)
print("PCPN DataFrame:\n", df_pcpn.head())

#### Check for missing data

In [ ]:
#check for missing data in temperature and precipitation DataFrames
#the parsed monthly columns are named Jan..Dec (not suffixed with _tmax/_pcpn)
month_cols = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
tmax_months = [col for col in month_cols if col in df_tmax.columns]
pcpn_months = [col for col in month_cols if col in df_pcpn.columns]

#values are still strings at this point (parsed straight from the raw text files),
#so convert to numeric before comparing against the missing-value sentinels
tmax_numeric = df_tmax[tmax_months].apply(pd.to_numeric, errors='coerce')
pcpn_numeric = df_pcpn[pcpn_months].apply(pd.to_numeric, errors='coerce')

#count missing values in temperature data (-99.90 indicates missing)
missing_tmax = (tmax_numeric == -99.90).sum()
print("Missing values in Temperature Data (per month):")
print(missing_tmax)

#count missing values in precipitation data (-9.99 indicates missing)
missing_pcpn = (pcpn_numeric == -9.99).sum()
print("\nMissing values in Precipitation Data (per month):")
print(missing_pcpn)

# Align with corn yield data

In [ ]:
#load crop yield data (path_df_step1 / filename set in the config cell above)
full_path = os.path.join(path_df_step1, filename)

df_corn_yield = pd.read_csv(
    full_path,
    dtype={
        'state_ansi': str,
        'county_ansi': str,
        'district_code': str
    }
)

print(df_corn_yield.head())

### Start pre-merging diagnostics

In [17]:
#filter the temperature and precipitation DataFrames to include only states in the corn yield data
#here we reuse the unique_states from the crop yield DataFrame as the allowed state list
allowed_states = df_corn_yield["state_ansi"].astype(str).unique()

df_tmax = df_tmax[df_tmax["state"].isin(allowed_states)]
df_pcpn = df_pcpn[df_pcpn["state"].isin(allowed_states)]

print("Filtered Temperature DataFrame States:", df_tmax["state"].unique())
print("Filtered Precipitation DataFrame States:", df_pcpn["state"].unique())

Filtered Temperature DataFrame States: []
Filtered Precipitation DataFrame States: []


In [15]:
#count unique counties per state in the temperature DataFrame:
tmax_counties_per_state = df_tmax.groupby('state')['county'].nunique()
print("Unique counties per state in df_tmax:")
print(tmax_counties_per_state)

#count unique counties per state in the corn yield DataFrame:
corn_counties_per_state = df_corn_yield.groupby('state_ansi')['county_ansi'].nunique()
print("\nUnique counties per state in df_corn_yield:")
print(corn_counties_per_state)

Unique counties per state in df_tmax:
Series([], Name: county, dtype: int64)

Unique counties per state in df_corn_yield:
state_ansi
17    102
18     92
19     99
27     85
31     89
Name: county_ansi, dtype: int64


In [16]:
#------------------------------------------------------------------
##--- CHECK HIGHEST MIN AND LOWEST MAX YEAR FOR CORN YIELD DATA ---
#compute the min and max year for each state in the corn yield dataset
state_year_ranges = df_corn_yield.groupby("state_ansi")["year"].agg(["min", "max"])
print("Year ranges per state:")
print(state_year_ranges)

#get the highest minimum year (i.e. the maximum of the minimum years)
highest_min_year = state_year_ranges["min"].max()

#get the lowest maximum year (i.e. the minimum of the maximum years)
lowest_max_year = state_year_ranges["max"].min()

print(f"Highest minimum year across states: {highest_min_year}")
print(f"Lowest maximum year across states: {lowest_max_year}")


#------------------------------------------------------------------
##--- APPLY TO WEATHER DATA ---
#convert the 'year' column to int (if not already) and filter the data frames
df_tmax["year"] = df_tmax["year"].astype(int)
df_tmax = df_tmax[(df_tmax["year"] >= highest_min_year) & (df_tmax["year"] <= lowest_max_year)]

df_pcpn["year"] = df_pcpn["year"].astype(int)
df_pcpn = df_pcpn[(df_pcpn["year"] >= highest_min_year) & (df_pcpn["year"] <= lowest_max_year)]

print("Filtered df_tmax:")
print(df_tmax.head(100000))
print("\nFiltered df_pcpn:")
print(df_pcpn.head(100000))

Year ranges per state:
             min   max
state_ansi            
17          1960  2024
18          1960  2024
19          1960  2024
27          1960  2024
31          1960  2024
Highest minimum year across states: 1960
Lowest maximum year across states: 2024
Filtered df_tmax:
Empty DataFrame
Columns: [raw_code, state, county, division, year, Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec]
Index: []

Filtered df_pcpn:
Empty DataFrame
Columns: [raw_code, state, county, division, year, Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec]
Index: []


### Make growing season variables

In [18]:
#be aware:
#- Get params from config file
#- convert to SI units

import calendar
#add parent directory to Python path
sys.path.append(os.path.abspath(".."))
#now import config
import config

#this creates the list of standard month abbreviations
#e.g., ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_name_list = calendar.month_abbr 

#uses the start and end month numbers from config to slice the list
#GS_START_MONTH (e.g., 4) becomes the start of the slice
#GS_END_MONTH + 1 (e.g., 9 + 1 = 10) becomes the end of the slice (exclusive)
months = list(month_name_list[config.GS_START_MONTH : config.GS_END_MONTH + 1])
print(months)

#--- Temperature Calculation ---
#convert the monthly temperature columns to numeric values
df_tmax[months] = df_tmax[months].apply(pd.to_numeric, errors='coerce')
#convert Fahrenheit to Celsius: °C = (°F - 32) × 5/9
df_tmax[months] = (df_tmax[months] - 32) * 5 / 9
#Compute the arithmetic mean (across the months) for each row
df_tmax['temp_gs'] = df_tmax[months].mean(axis=1)
#group by year, state, county, and division to get one value per group
df_temp_growing_season = df_tmax.groupby(
    ['year', 'state', 'county', 'division']
)['temp_gs'].mean().reset_index()

print("Growing season temperature by year, state, county, and division:")
print(df_temp_growing_season.head())

#--- Precipitation Calculation ---
#convert the monthly precipitation columns to numeric values
df_pcpn[months] = df_pcpn[months].apply(pd.to_numeric, errors='coerce')
#convert inches to millimeters: mm = inches × 25.4
df_pcpn[months] = df_pcpn[months] * 25.4
#compute the total precipitation (sum over the months) for each row
df_pcpn['pcpn_gs'] = df_pcpn[months].sum(axis=1)
#group by year, state, county, and division to get one value per group
df_precip_growing_season = df_pcpn.groupby(
    ['year', 'state', 'county', 'division']
)['pcpn_gs'].mean().reset_index()

print("\nGrowing season precipitation by year, state, county, and division:")
print(df_precip_growing_season.head())

['Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep']
Growing season temperature by year, state, county, and division:
Empty DataFrame
Columns: [year, state, county, division, temp_gs]
Index: []

Growing season precipitation by year, state, county, and division:
Empty DataFrame
Columns: [year, state, county, division, pcpn_gs]
Index: []


#### Check one more time if nothing is missing

In [19]:
#use the corn yield year range as the common complete years
complete_years = set(range(highest_min_year, lowest_max_year + 1))

#get the unique (state, county) keys present in either temperature or precipitation datasets
keys_temp = set(df_temp_growing_season.groupby(["state", "county"]).groups.keys())
keys_pcpn = set(df_precip_growing_season.groupby(["state", "county"]).groups.keys())
all_keys = keys_temp.union(keys_pcpn)

missing_weather_report = {}

for key in all_keys:
    state, county = key
    #get years available from the temperature dataset for the county (if any)
    years_temp = set(
        df_temp_growing_season.loc[
            (df_temp_growing_season["state"] == state) & (df_temp_growing_season["county"] == county),
            "year"
        ].astype(int).unique()
    )
    #get years available from the precipitation dataset for the county (if any)
    years_pcpn = set(
        df_precip_growing_season.loc[
            (df_precip_growing_season["state"] == state) & (df_precip_growing_season["county"] == county),
            "year"
        ].astype(int).unique()
    )
    
    #combine the available years from both data types
    years_present = years_temp.union(years_pcpn)
    missing_rows = sorted(complete_years - years_present)
    
    if missing_rows:
        missing_weather_report[(state, county)] = missing_rows

#print the missing report in sorted order if any missing data exists, else print 'no missing data'
if missing_weather_report:
    for state, county in sorted(missing_weather_report.keys()):
        print(f"State {state}, County {county}:")
        print(f"  Rows missing for years: {missing_weather_report[(state, county)]}")
else:
    print("no missing data")

no missing data


### Merge data

In [20]:
#example: load the dataframes (swap in the real file loading here)
# corn_yield_df = pd.read_csv("corn_yield.csv")
# df_temp_growing_season = pd.read_csv("temp_growing_season.csv")
# df_precip_growing_season = pd.read_csv("precip_growing_season.csv")

#rename columns in corn_yield_df to match the keys in the other dataframes
corn_yield_df = df_corn_yield.rename(columns={
    'state_ansi': 'state',
    'county_ansi': 'county',
    'district_code': 'division'
}).copy()

#ensure that the key columns 'state', 'county', and 'year' have the same data types
#in this example, we'll convert 'state' and 'county' to string.
for df in [corn_yield_df, df_temp_growing_season, df_precip_growing_season]:
    df['state'] = df['state'].astype(str)
    df['county'] = df['county'].astype(str)
    #assuming 'year' is consistent (e.g., int) between datasets; if not, convert as needed:
    # df['year'] = df['year'].astype(int)

#merge corn_yield_df with the temperature dataframe using an outer join.
merged_df = pd.merge(
    corn_yield_df,
    df_temp_growing_season,
    on=['state', 'county', 'year'],
    how='outer'
)

#merge the resulting dataframe with the precipitation dataframe using an outer join.
merged_df = pd.merge(
    merged_df,
    df_precip_growing_season,
    on=['state', 'county', 'year'],
    how='outer'
)

#display the first few rows of the merged dataframe
print(merged_df.head())

   year state county division_x                                data_item  \
0  1960    17    001         03  SOYBEANS - YIELD, MEASURED IN BU / ACRE   
1  1961    17    001         03  SOYBEANS - YIELD, MEASURED IN BU / ACRE   
2  1962    17    001         03  SOYBEANS - YIELD, MEASURED IN BU / ACRE   
3  1963    17    001         03  SOYBEANS - YIELD, MEASURED IN BU / ACRE   
4  1964    17    001         03  SOYBEANS - YIELD, MEASURED IN BU / ACRE   

   value  cv division_y temp_gs division pcpn_gs  
0   23.5 NaN        NaN     NaN      NaN     NaN  
1   27.0 NaN        NaN     NaN      NaN     NaN  
2   27.5 NaN        NaN     NaN      NaN     NaN  
3   29.0 NaN        NaN     NaN      NaN     NaN  
4   28.0 NaN        NaN     NaN      NaN     NaN  


##### Fix the triple division presence

In [21]:
#check only rows where division_x is not NaN
mask = merged_df["division_x"].notna()

#evaluate whether division_x equals division_y in those rows
if (merged_df.loc[mask, "division_x"] == merged_df.loc[mask, "division_y"]).all():
    print("All non-NaN division_x values match division_y.")
else:
    print("Mismatch found between division_x and division_y in some rows.")
    #optionally, print rows with mismatches for inspection:
    mismatches = merged_df.loc[mask][merged_df.loc[mask, "division_x"] != merged_df.loc[mask, "division_y"]]
    print(mismatches[["division_x", "division_y"]])

Mismatch found between division_x and division_y in some rows.
      division_x division_y
0             03        NaN
1             03        NaN
2             03        NaN
3             03        NaN
4             03        NaN
...          ...        ...
27897         06        NaN
27898         06        NaN
27899         06        NaN
27900         06        NaN
27901         06        NaN

[27902 rows x 2 columns]


After inspection of the df it was found that for state 17, county 199, the division is 08 for the yield data and 09 for the NOAA data. A quick inspection of the 'county-to-climdivs' and '240917_corn_yield_data' shows that this is the case from the start and not a coding error.

In [22]:
#check if all non-NaN values in division_y match division
mask_div = merged_df["division_y"].notna()

if (merged_df.loc[mask_div, "division_y"] == merged_df.loc[mask_div, "division"]).all():
    print("All non-NaN values in division_y match division.")
else:
    print("Mismatch found between division_y and division in some rows.")
    mismatches = merged_df.loc[mask_div][
        merged_df.loc[mask_div, "division_y"] != merged_df.loc[mask_div, "division"]
    ]
    print(mismatches[["division_y", "division"]])

All non-NaN values in division_y match division.


##### Neatly finalize divisions

In [23]:
#assume merged_df is already created via prior merging steps

#create a copy of merged_df and work with the new DataFrame final_merged_df
final_merged_df = merged_df.copy()

#1. Rename 'division_x' to 'division_yield'
final_merged_df.rename(columns={'division_x': 'division_yield'}, inplace=True)

#2. Create new column 'division_noaa'
#since you've verified that division_y and division are essentially the same,
#we fill from division_y and fallback to division if necessary.
final_merged_df['division_noaa'] = final_merged_df['division_y'].fillna(final_merged_df['division'])

#3. Drop the now redundant columns 'division_y' and 'division'
final_merged_df.drop(columns=['division_y', 'division'], inplace=True)

#4. Reorder columns so that 'division_noaa' is placed immediately after 'division_yield'
cols = list(final_merged_df.columns)
#find index of 'division_yield'
idx = cols.index('division_yield')
#build the new column order:
new_cols = cols[:idx+1] + ['division_noaa'] + cols[idx+1:]
#in case 'division_noaa' appears twice, remove duplicates while preserving order
new_cols = list(dict.fromkeys(new_cols))
final_merged_df = final_merged_df[new_cols]

#display the first few rows to verify the ordering
print(final_merged_df.head())

   year state county division_yield  division_noaa  \
0  1960    17    001             03            NaN   
1  1961    17    001             03            NaN   
2  1962    17    001             03            NaN   
3  1963    17    001             03            NaN   
4  1964    17    001             03            NaN   

                                 data_item  value  cv temp_gs pcpn_gs  
0  SOYBEANS - YIELD, MEASURED IN BU / ACRE   23.5 NaN     NaN     NaN  
1  SOYBEANS - YIELD, MEASURED IN BU / ACRE   27.0 NaN     NaN     NaN  
2  SOYBEANS - YIELD, MEASURED IN BU / ACRE   27.5 NaN     NaN     NaN  
3  SOYBEANS - YIELD, MEASURED IN BU / ACRE   29.0 NaN     NaN     NaN  
4  SOYBEANS - YIELD, MEASURED IN BU / ACRE   28.0 NaN     NaN     NaN  


C:\Users\rickg\AppData\Local\Temp\ipykernel_14000\1056427802.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_merged_df['division_noaa'] = final_merged_df['division_y'].fillna(final_merged_df['division'])


In [24]:
##--- Check if the variable values of the df with new division set-up are the same as before ---

#define the columns to check
cols_to_check = ['value', 'temp_gs', 'pcpn_gs']

#compare the corresponding columns between merged_df and final_merged_df
are_equal = merged_df[cols_to_check].equals(final_merged_df[cols_to_check])
print("The 'value', 'temp_gs', and 'pcpn_gs' columns are the same in both DataFrames:", are_equal)

The 'value', 'temp_gs', and 'pcpn_gs' columns are the same in both DataFrames: True


### Save the created df containing both crop yield data and weather regressors.

In [ ]:
#convert key columns to string to preserve leading zeros!!!!
for col in ['county', 'division_yield', 'division_noaa']:
    final_merged_df[col] = final_merged_df[col].astype(str)

#save_file set in the config cell above
final_merged_df.to_csv(save_file, index=False, quoting=csv.QUOTE_ALL)

print(f"Data saved to {save_file}")